In [1]:
import sys
from pathlib import Path

src_path = Path("../src").resolve()
sys.path.insert(0, str(src_path))

print(src_path)

C:\Users\abeyt\Downloads\nem-bess-optimisation\src


In [2]:
import battery

print(battery.__file__)
print(battery.POWER_MW)
print(battery.CAPACITY_MWH)
print(battery.INITIAL_SOC_MWH)

C:\Users\abeyt\Downloads\nem-bess-optimisation\src\battery.py
100.0
200.0
100.0


In [3]:
from baseline import run_baseline
import pandas as pd

prices = pd.read_csv(
    "../data/processed/vic_spot_prices_2025_07.csv",
    parse_dates=["SETTLEMENTDATE"],
)

prices.head()

,SETTLEMENTDATE,REGIONID,RRP
0,2025-07-01 00:05:00,VIC1,176.61966
1,2025-07-01 00:10:00,VIC1,182.30989
2,2025-07-01 00:15:00,VIC1,169.47610
3,2025-07-01 00:20:00,VIC1,184.67607
4,2025-07-01 00:25:00,VIC1,185.73900


In [4]:
charge_threshold = prices["RRP"].quantile(0.25)
discharge_threshold = prices["RRP"].quantile(0.75)

print(f"Charge threshold: ${charge_threshold:.2f}/MWh")
print(f"Discharge threshold: ${discharge_threshold:.2f}/MWh")

Charge threshold: $8.95/MWh
Discharge threshold: $135.44/MWh


In [5]:
baseline_results = run_baseline(
    prices,
    charge_threshold,
    discharge_threshold,
)

baseline_results.head()

,SETTLEMENTDATE,RRP,charge_mw,discharge_mw,soc_mwh,revenue
0,2025-07-01 00:05:00,176.61966,0.0,100.0,91.215895,1471.830500
1,2025-07-01 00:10:00,182.30989,0.0,100.0,82.431791,1519.249083
2,2025-07-01 00:15:00,169.47610,0.0,100.0,73.647686,1412.300833
3,2025-07-01 00:20:00,184.67607,0.0,100.0,64.863582,1538.967250
4,2025-07-01 00:25:00,185.73900,0.0,100.0,56.079477,1547.825000


In [6]:
total_revenue = baseline_results["revenue"].sum()

print(f"Baseline July revenue: ${total_revenue:,.2f}")

Baseline July revenue: $521,874.69


In [7]:
baseline_results["soc_mwh"].agg(["min", "max"])

min     20.0
max    180.0
Name: soc_mwh, dtype: float64

In [9]:
from pathlib import Path

RESULTS_DIR = Path("../outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
baseline_results.to_csv(
    RESULTS_DIR / "baseline_dispatch.csv",
    index=False
)

In [ ]:
baseline_summary = pd.DataFrame({
    "strategy": ["Rule-based"],
    "revenue": [baseline_results["revenue"].sum()],
    "min_soc_mwh": [baseline_results["soc_mwh"].min()],
    "max_soc_mwh": [baseline_results["soc_mwh"].max()]
})


In [ ]:
from pathlib import Path

RESULTS_DIR = Path("../outputs/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results.to_csv(
    RESULTS_DIR / "baseline_dispatch.csv",
    index=False
)

print("Saved baseline_dispatch.csv")